## Change of variable

Suppose we have a posterior distribution of $\theta_1,\ \theta_2$ given by
$$
p(\theta_1, \theta_2 \mid d) 
$$
and we would like to transform it into the distribution of $\vartheta_1,\ \vartheta_2$, then by the change of variable formula, it amounts to:
$$
p(\vartheta_1, \vartheta_2 \mid d) = 
\vert J \vert \,
p(\theta_1, \theta_2 \mid d)
$$
where $\vert J \vert $ is the Jacobian:
$$
\vert J \vert = 
\begin{vmatrix}
\frac{\partial \theta_1}{\partial \vartheta_1} & \frac{\partial \theta_2}{\partial \vartheta_1} \\
\frac{\partial \theta_1}{\partial \vartheta_2} & \frac{\partial \theta_2}{\partial \vartheta_2} \\
\end{vmatrix}
$$

In [2]:
import jax
jax.config.update("jax_enable_x64", True)
from jax.lax import integer_pow
from gwfast.gwfastGlobals import DAY_TO_SEC
from gwfast.lensing_utils_alt import compute_lensed_angles_approx

In [4]:
def lensing_transform(lensing_parameters):
    outputs = compute_lensed_angles_approx(lensing_parameters)
    phenom_changes = {}
    phenom_changes['delta_incl'] = outputs['iota_p'] - outputs['iota_m']
    phenom_changes['delta_phi'] = outputs['phase_p'] - outputs['phase_m']

    # (Radial gravitational potential is cancelled)
    phenom_changes['relative_mass'] = (1 + outputs['z_rel_p']) / (1 + outputs['z_rel_m'])
    relative_magification = outputs['sqrt_mu_m'] / outputs['sqrt_mu_p']
    phenom_changes['relative_distance'] = relative_magification * integer_pow((1 + outputs['z_rel_p']) / (1 + outputs['z_rel_m']), 2)
    phenom_changes['delta_time'] = outputs['delta_time'] / DAY_TO_SEC
    return phenom_changes


In [7]:
jax.jacfwd(lensing_transform)({
    'iota': np.pi/2 * 0.97, 'phase': 0.4, 'R_orbit': 300., 'src_pos': 0.1, 
    'M_lz': 1e6, 'dL': 1.
})

{'delta_incl': {'M_lz': Array(1.12475993e-21, dtype=float64),
  'R_orbit': Array(0.00012771, dtype=float64),
  'dL': Array(-1.29150524e-15, dtype=float64),
  'iota': Array(1.62842118, dtype=float64),
  'phase': Array(0., dtype=float64),
  'src_pos': Array(0.77397965, dtype=float64)},
 'delta_phi': {'M_lz': Array(2.11914882e-21, dtype=float64),
  'R_orbit': Array(0.00024061, dtype=float64),
  'dL': Array(-2.43331198e-15, dtype=float64),
  'iota': Array(-0.85941522, dtype=float64),
  'phase': Array(0., dtype=float64),
  'src_pos': Array(-0.41171121, dtype=float64)},
 'delta_time': {'M_lz': Array(-4.95383508e-10, dtype=float64),
  'R_orbit': Array(-9.14259276e-07, dtype=float64),
  'dL': Array(7.34404525e-05, dtype=float64),
  'iota': Array(-0., dtype=float64),
  'phase': Array(-0., dtype=float64),
  'src_pos': Array(-0.00551326, dtype=float64)},
 'relative_distance': {'M_lz': Array(3.80053695e-21, dtype=float64),
  'R_orbit': Array(-0.00052739, dtype=float64),
  'dL': Array(3.20865654e-1